In [ ]:
import xarray as xr

In [ ]:
file = "GFS0p25/gfs.t00z.pgrb2.0p25.f000"
ds_surface = xr.open_dataset(
    file,
    engine="cfgrib",
    backend_kwargs={"filter_by_keys": {"typeOfLevel": "surface"}, "indexpath": ""}
)
ds_surface

In [ ]:
orog = ds_surface["orog"]
orog = orog.drop_vars(['time','step','surface','valid_time'])
orog.to_netcdf("GFS0p25/gfs_full_orography.nc")

In [ ]:
orog.sel(latitude=slice(48,38),longitude=slice(278,292)).plot.contourf()

In [ ]:
orog_nys_cropped = orog.sel(latitude=slice(48,38),longitude=slice(278,292))
orog_nys_cropped.to_netcdf("GFS0p25/gfs_nys_cropped_orography.nc")

In [ ]:
lsm = ds_surface["lsm"]
lsm = lsm.drop_vars(['time','step','surface','valid_time'])
lsm.to_netcdf("GFS0p25/gfs_full_land_sea_mask.nc")

In [ ]:
lsm_nys_cropped = lsm.sel(latitude=slice(48,38),longitude=slice(278,292))
lsm_nys_cropped.to_netcdf("GFS0p25/gfs_nys_cropped_lsm.nc")

In [ ]:
gfs_nys_cropped = xr.open_dataset("GFS0p25/gfs_nys_cropped_orography.nc")
urma_nys = xr.open_dataset("urma_nys_orography.nc")
lsm_nys_cropped = xr.open_dataset("GFS0p25/gfs_nys_cropped_lsm.nc")

In [ ]:
import xesmf as xe
# We don't have to provide grid dicts, rather source files having the latitude/lat and longitude/lon coordinates can work.
regridder_gfs_to_urma = xe.Regridder(
    gfs_nys_cropped,
    urma_nys,
    method='bilinear',
    filename='GFS0p25/xesmf_weights_gfs_to_urma.nc',
    reuse_weights=False
)

In [ ]:
regridder_gfs_to_urma = xe.Regridder(
    gfs_nys_cropped,
    urma_nys,
    method='bilinear',
    filename='GFS0p25/xesmf_weights_gfs_to_urma.nc',
    reuse_weights=True
)

In [ ]:
regridder_gfs_to_urma(gfs_nys_cropped.orog).plot.contourf(x='longitude', y='latitude')

In [ ]:
regridder_gfs_to_urma(lsm_nys_cropped.lsm).plot.contourf(x='longitude', y='latitude')